In [ ]:
!pip install vllm openai gymnasium minigrid imageio

In [ ]:
import subprocess
import time

model_name = "Qwen/Qwen2.5-7B-Instruct"

print(f"Loading vLLM server with model: {model_name}")
print("Startup takes about 3–4 minutes...")

command = f"nohup python -m vllm.entrypoints.openai.api_server \
--model {model_name} \
--dtype auto \
--api-key empty \
--port 8000 \
--gpu-memory-utilization 0.9 \
> vllm_server.log 2>&1 &"

subprocess.run(command, shell=True)

In [ ]:
import requests
import time

url = "http://localhost:8000/v1/models"
headers = {"Authorization": "Bearer empty"}

for i in range(20):
    try:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            print("✅ vLLM loaded successfully")
            print("Model:", response.json()['data'][0]['id'])
            break
    except requests.exceptions.ConnectionError:
        print(f"[{i+1}/20] Server not ready, retrying...")
        time.sleep(10)
else:
    print("❌ Server failed to start. Check vllm_server.log")

In [ ]:
import gymnasium as gym
from minigrid.wrappers import FullyObsWrapper
from minigrid.core.actions import Actions


class MinigridTextWrapper:

    def __init__(self, env_id, render_mode=None):
        self.env = gym.make(env_id, render_mode=render_mode)
        self.env = FullyObsWrapper(self.env)
        self.action_map = {
            "turn_left": Actions.left,
            "turn_right": Actions.right,
            "move_forward": Actions.forward,
        }

    def _base(self):
        return self.env.unwrapped

    def get_text_obs(self, obs):
        base = self._base()
        ax, ay = int(base.agent_pos[0]), int(base.agent_pos[1])
        agent_dir = int(base.agent_dir)
        dirs = ["right", "down", "left", "up"]
        facing = dirs[agent_dir]
        desc = f"Agent at [{ax},{ay}] facing {facing}. "

        front_obj = None
        if hasattr(base, "front_pos"):
            fx, fy = int(base.front_pos[0]), int(base.front_pos[1])
            front_obj = base.grid.get(fx, fy)

        if front_obj is None:
            desc += "The cell directly in front of you is empty. "
        else:
            desc += f"The cell directly in front of you contains {front_obj.type}. "

        grid = base.grid
        objects = []
        for x in range(grid.width):
            for y in range(grid.height):
                cell = grid.get(x, y)
                if cell is None:
                    continue
                obj_type = getattr(cell, "type", None)
                if obj_type in ("lava", "goal", "wall"):
                    objects.append(f"{obj_type} at [{x},{y}]")
        if objects:
            desc += "Objects: " + ", ".join(objects) + "."
        return desc

    def reset(self):
        obs, _ = self.env.reset()
        return self.get_text_obs(obs)

    def step(self, action_str):
        action = self.action_map.get(action_str.lower(), Actions.forward)
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return self.get_text_obs(obs), reward, done, info

In [ ]:
import imageio
import os

def evaluate_agent(agent,
                   env_name="MiniGrid-LavaGapS6-v0",
                   num_episodes=10,
                   max_steps_per_episode=100,
                   gif_folder="episode_gifs"):

    print(f"Evaluating {env_name} for {num_episodes} episodes")

    env = MinigridTextWrapper(env_name, render_mode="rgb_array")

    os.makedirs(gif_folder, exist_ok=True)

    metrics = {
        "success_count": 0,
        "total_steps_success": 0,
        "total_inference_time": 0,
        "total_tokens": 0,
        "total_actions": 0
    }

    for episode in range(num_episodes):

        obs = env.reset()
        agent.reset()

        done = False
        step_count = 0
        episode_reward = 0

        frames = []

        # record initial frame
        frames.append(env.env.render())

        while not done and step_count < max_steps_per_episode:

            action, response, inf_time = agent.act(obs)

            metrics["total_inference_time"] += inf_time
            metrics["total_actions"] += 1

            obs, reward, done, _ = env.step(action)

            step_count += 1
            episode_reward += reward

            # record frame
            frames.append(env.env.render())

        success = episode_reward > 0

        if success:
            metrics["success_count"] += 1
            metrics["total_steps_success"] += step_count

        # save GIF
        gif_path = f"{gif_folder}/episode_{episode+1}.gif"
        imageio.mimsave(gif_path, frames, fps=5)

        print(f"Episode {episode+1}/{num_episodes} | Success: {success} | Steps: {step_count} | GIF saved: {gif_path}")

    success_rate = metrics["success_count"] / num_episodes * 100

    avg_steps = (
        metrics["total_steps_success"] / metrics["success_count"]
        if metrics["success_count"] > 0 else float("inf")
    )

    avg_inf_time = metrics["total_inference_time"] / metrics["total_actions"]

    print("\n" + "="*40)
    print("🏆 Final Metrics")
    print("="*40)
    print(f"Success Rate: {success_rate:.2f}%")
    print(f"Avg Steps (successful episodes): {avg_steps:.2f}")
    print(f"Avg Inference Time per step: {avg_inf_time:.4f} sec")
    print("="*40)

    return metrics

In [ ]:
import matplotlib.pyplot as plt

def debug_agent(agent, steps=10):

    env = MinigridTextWrapper("MiniGrid-LavaGapS6-v0", render_mode="rgb_array")

    obs = env.reset()
    agent.reset()

    for step in range(steps):

        print(f"\nSTEP {step+1}")

        print("\nOBSERVATION:\n", obs)

        prompt = agent.build_prompt(obs)
        print("\nPROMPT:\n", prompt)

        action, response, latency = agent.act(obs)

        print("\nLLM RESPONSE:\n", response)
        print("\nACTION:", action)

        frame = env.env.render()

        plt.imshow(frame)
        plt.axis("off")
        plt.show()

        obs, reward, done, _ = env.step(action)

        if done:
            print("\nEpisode finished | reward:", reward)
            break

    env.env.close()

In [ ]:
import torch
import re
import time
from collections import deque
from dataclasses import dataclass
from transformers import AutoTokenizer, AutoModelForCausalLM

@dataclass
class ThoughtTemplate:
    name: str
    description: str
    reasoning_pattern: str
    usage_count: int = 0
    success_rate: float = 0.0
    def instantiate(self, distilled_obs):
        context = []
        if distilled_obs.get('agent_pos'): context.append(f"You are at {distilled_obs['agent_pos']} facing {distilled_obs.get('facing', 'unknown')}.")
        if distilled_obs.get('target_pos'): context.append(f"Target is at {distilled_obs['target_pos']}.")
        if distilled_obs.get('front_object'): context.append(f"In front is a {distilled_obs['front_object']}.")
        if distilled_obs.get('nearby_objects'): context.append(f"Nearby: {', '.join(distilled_obs['nearby_objects'])}.")
        return f"[TEMPLATE: {self.name}]\nContext: {' '.join(context)}\nPattern: {self.reasoning_pattern}"

ALL_TEMPLATES = [
    ThoughtTemplate("Direct Navigation", "Visible target.", "1. Locate 2. Align 3. Move"),
    ThoughtTemplate("Obstacle Avoidance", "Detour around lava/walls.", "1. Identify blocker 2. Detour plan 3. Move"),
    ThoughtTemplate("Object Acquisition", "Pickup key/ball.", "1. Scan 2. Navigate 3. Face 4. Pickup"),
    ThoughtTemplate("Unlock Door", "Unlock and open door.", "1. Move to door 2. Face 3. Toggle"),
    ThoughtTemplate("Clear Path", "Move object aside.", "1. Identify 2. Face 3. Pickup 4. Drop aside"),
]

class MetaBuffer:
    def __init__(self, size=5):
        self.templates = {t.name: t for t in ALL_TEMPLATES[:size]}
    def retrieve(self, d_dict):
        desc = str(d_dict).lower()
        best, max_s = None, -1.0
        for n, t in self.templates.items():
            score = (t.success_rate if t.usage_count > 0 else 0.5)
            if (('lava' in n.lower() and 'lava' in desc) or ('door' in n.lower() and 'door' in desc)): score += 1.0
            if score > max_s: max_s, best = score, t
        return best

class ProblemDistiller:
    @staticmethod
    def distill(obs):
        d = {'agent_pos':None, 'facing':None, 'target_pos':None, 'front_object':None, 'nearby_objects':[]}
        m = re.search(r"Agent (?:is )?at \[\s*(\d+)\s*,\s*(\d+)\s*\] facing (\w+)", obs)
        if m: d['agent_pos'], d['facing'] = [int(m.group(1)), int(m.group(2))], m.group(3)
        m = re.search(r"(?:Goal|Target|goal|target).*?(?:is )?at \[\s*(\d+)\s*,\s*(\d+)\s*\]", obs)
        if m: d['target_pos'] = [int(m.group(1)), int(m.group(2))]
        m = re.search(r"(?:In front of you is a|cell directly in front.*?contains) (.*?)\.", obs)
        if m: d['front_object'] = m.group(1).strip()
        return d

class LLMClient:
    def __init__(self, model_name, mock=False):
        self.model_name, self.mock = model_name, mock
        self.model, self.tok = None, None
        if not mock:
            self.tok = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype='auto', device_map='auto')
            self.model.eval()
    def query(self, sys, user):
        s = time.time()
        if self.mock: 
            res = 'move_forward' if 'lava' not in user.lower() else 'turn_left'
            tokens = 0
        else:
            p = self.tok.apply_chat_template([{'role':'system','content':sys},{'role':'user','content':user}], tokenize=False, add_generation_prompt=True)
            i = self.tok(p, return_tensors='pt').to(self.model.device)
            with torch.no_grad(): out = self.model.generate(**i, max_new_tokens=16)
            res = self.tok.decode(out[0][i['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
            tokens = int(out[0][i['input_ids'].shape[-1]:].shape[-1])
        return res, tokens, time.time()-s

class BoTAgent:
    def __init__(self, model='Qwen/Qwen2.5-7B-Instruct', mock=False, h_size=3):
        self.llm = LLMClient(model, mock=mock)
        self.buffer = MetaBuffer()
        self.h_size, self.memory = h_size, []
    def act(self, obs):
        dist = ProblemDistiller.distill(obs)
        t = self.buffer.retrieve(dist)
        aid = t.instantiate(dist) if t else ''
        hist = self.memory[-self.h_size:] if self.h_size > 0 else []
        res, _, lat = self.llm.query('You are a MiniGrid agent. Say: move_forward, turn_left, or turn_right.', f'Obs: {obs}\n{aid}\nHistory: {hist}\nAction:')
        act = 'move_forward'
        if 'left' in res.lower(): act = 'turn_left'
        elif 'right' in res.lower(): act = 'turn_right'
        self.memory.append((obs, act))
        return act, res, lat
    def reset(self): self.memory = []


In [ ]:
env = MinigridTextWrapper("MiniGrid-LavaGapS6-v0")
agent = BoTAgent()

In [ ]:
print('\n--- Evaluating on MiniGrid-Empty-8x8-v0 ---')
evaluate_agent(agent,
               env_name="MiniGrid-Empty-8x8-v0",
               num_episodes=10,
               max_steps_per_episode=100)

In [ ]:
print('\n--- Evaluating on MiniGrid-LavaGapS6-v0 ---')
evaluate_agent(agent,
               env_name="MiniGrid-LavaGapS6-v0",
               num_episodes=10,
               max_steps_per_episode=100)

In [ ]:
print('\n--- Evaluating on MiniGrid-LavaCrossingS9N2-v0 ---')
evaluate_agent(agent,
               env_name="MiniGrid-LavaCrossingS9N2-v0",
               num_episodes=10,
               max_steps_per_episode=100)